# Full-Mamba + TCN Temporal Attention Pooling — BCI IV-2a LOSO


In [ ]:
# Enable GPU and Internet in Kaggle, then run this cell.
BRANCH = 'feature/hada-full-mamba-tcn-attention-pooling'
!pip -q install uv
%cd /kaggle/working
!if [ -d tcformer-test/.git ]; then git -C tcformer-test fetch origin $BRANCH && git -C tcformer-test checkout $BRANCH && git -C tcformer-test pull --ff-only origin $BRANCH; else git clone --branch $BRANCH --single-branch https://github.com/CuongDM1806/tcformer-test.git; fi
%cd /kaggle/working/tcformer-test
!git log -1 --oneline
!grep -nE 'temporal_attention|pool_mix_logit' models/tcformer.py
!uv venv --clear --python 3.10 .venv
!uv pip install --python .venv/bin/python torch==2.7.1 torchvision==0.22.1 --index-url https://download.pytorch.org/whl/cu126
!uv pip install --python .venv/bin/python -r requirements.txt
!nvidia-smi
!CUDA_VISIBLE_DEVICES=0 .venv/bin/python -c "import torch; from models.tcformer import TCNHead; m=TCNHead(64,4,2,4,0.3,4).cuda(); x=torch.randn(2,64,20,device='cuda',requires_grad=True); y=m.pool_temporal_features(m.extract_temporal_features(x)); assert y.shape==(2,64); y.mean().backward(); print('Attention-pooling smoke:',tuple(y.shape))"
!PYTHONUNBUFFERED=1 MPLBACKEND=Agg .venv/bin/python -u train_pipeline.py --model hada_tcformer --dataset bcic2a --loso --gpu_id 0
